# 01 · Data understanding

Goal: describe the raw UCI *Parking Birmingham* file exactly as it is, before any cleaning, and confirm it matches the official description (35,717 rows, 4 attributes, 4 Oct – 19 Dec 2016).

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from IPython.display import Image, display
from src import config as C

In [2]:
from src.data_preprocessing import load_raw, quality_report
raw = load_raw()
print(raw.shape)
raw.head()

(35717, 4)


,SystemCodeNumber,Capacity,Occupancy,LastUpdated
0,BHMBCCMKT01,577,61,2016-10-04 07:59:42
1,BHMBCCMKT01,577,64,2016-10-04 08:25:42
2,BHMBCCMKT01,577,80,2016-10-04 08:59:42
3,BHMBCCMKT01,577,107,2016-10-04 09:32:46
4,BHMBCCMKT01,577,150,2016-10-04 09:59:48


In [3]:
raw.dtypes

SystemCodeNumber               str
Capacity                     int64
Occupancy                    int64
LastUpdated         datetime64[us]
dtype: object

## Audit
Missing cells, duplicates, impossible values and coverage, computed directly from the file.

In [4]:
q = quality_report(raw)
{k: v for k, v in q.items() if k != 'per_park'}

{'rows': 35717,
 'columns': ['SystemCodeNumber', 'Capacity', 'Occupancy', 'LastUpdated'],
 'car_parks': 30,
 'first_timestamp': '2016-10-04 07:46:28',
 'last_timestamp': '2016-12-19 16:30:35',
 'calendar_days_in_range': 77,
 'days_with_data': 73,
 'days_without_any_data': ['2016-10-20',
  '2016-10-21',
  '2016-12-03',
  '2016-12-04'],
 'missing_values_per_column': {'SystemCodeNumber': 0,
  'Capacity': 0,
  'Occupancy': 0,
  'LastUpdated': 0},
 'missing_value_pct_per_column': {'SystemCodeNumber': 0.0,
  'Capacity': 0.0,
  'Occupancy': 0.0,
  'LastUpdated': 0.0},
 'exact_duplicate_rows': 216,
 'duplicate_park_timestamp_pairs': 216,
 'negative_occupancy_rows': 12,
 'occupancy_above_capacity_rows': 373,
 'max_occupancy_rate': 1.0413,
 'min_occupancy_rate': -0.0167,
 'parks_with_changing_capacity': 0,
 'median_minutes_between_readings': 29.98,
 'typical_reading_hours': '07:30:33 - 16:34:28'}

In [5]:
per_park = pd.DataFrame(q['per_park']).T.sort_values('observations')
per_park

,observations,capacity,first_reading,last_reading,mean_occupancy_rate
BHMBRTARC01,88,496,2016-12-13 08:02:51,2016-12-18 16:30:25,0.7766
NIA North,162,480,2016-10-16 08:01:13,2016-11-30 16:28:40,0.0738
BHMNCPNHS01,1038,500,2016-10-04 07:59:42,2016-12-19 16:30:35,0.711
Bull Ring,1186,3053,2016-10-04 07:59:42,2016-12-19 16:30:35,0.4765
BHMBRCBRG03,1186,849,2016-10-04 07:59:42,2016-12-19 16:30:35,0.2867
BHMNCPRAN01,1186,600,2016-10-04 07:59:42,2016-12-19 16:30:35,0.6455
BHMBRCBRG01,1186,1010,2016-10-04 07:59:42,2016-12-19 16:30:35,0.641
BHMBRCBRG02,1186,1194,2016-10-04 07:59:42,2016-12-19 16:30:35,0.4725
NIA Car Parks,1204,1268,2016-10-04 07:59:42,2016-12-16 16:30:12,0.1635
NIA South,1204,788,2016-10-04 07:59:42,2016-12-16 16:30:12,0.2492


In [6]:
rate = raw['Occupancy'] / raw['Capacity']
rate.describe()

count    35717.000000
mean         0.493338
std          0.271353
min         -0.016667
25%          0.255881
50%          0.471872
75%          0.717504
max          1.041344
dtype: float64

In [7]:
print('readings per hour of day');print(raw['LastUpdated'].dt.hour.value_counts().sort_index())

readings per hour of day
LastUpdated
7      834
8     4018
9     4037
10    3829
11    3981
12    3855
13    4139
14    3945
15    3942
16    3137
Name: count, dtype: int64


## What this tells us
* The file matches the UCI description: 35,717 rows, the four documented columns, 30 car parks, no empty cells.
* *No empty cells* does not mean *no missing data*: 216 rows are exact duplicates, four calendar days have no readings at all (20–21 Oct, 3–4 Dec), and two car parks (BHMBRTARC01 with 88 readings, NIA North with 162) report only rarely.
* 12 readings have negative occupancy (physically impossible → sensor error) and 373 exceed capacity (up to 104%; cars queuing or counter drift).
* Capacity never changes for a car park, so it can be used to turn a predicted rate into predicted free spaces.
* Readings arrive about every 27–33 minutes between ~08:00 and 16:30, which fixes a natural 30-minute time step and means horizons shorter than 30 minutes cannot be supported.